In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import * 


In [0]:
silver_table = "frauddetection.silver.transactions"
bornze_table = "frauddetection.bronze.transactions"
gold_table_user_behavior = "frauddetection.gold.user_transactions_behavior"

In [0]:
silver_df = spark.read.table(silver_table)



In [0]:
display(silver_df.limit(5))

In [0]:
user_behavior_df = silver_df.groupBy("customer_id").agg(
    round(max(col("amount")),2).alias("max_amount"),
    round(min(col("amount")),2).alias("min_amount"),
    round(avg(col("amount")),2).alias("avg_amount"),
    count("*").alias("transactions_count"),
    round(stddev(col("amount")),2).alias("std_amount")
)

In [0]:
display(user_behavior_df.limit(10))

In [0]:
user_behavior_df = user_behavior_df.dropDuplicates(
    subset = ["customer_id"]
)

In [0]:
user_behavior_df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(gold_table_user_behavior)

In [0]:
%sql
select * from frauddetection.gold.user_transactions_behavior limit 10

In [0]:
user_top_location_table = "frauddetection.gold.user_locations"

In [0]:
user_top_locations = silver_df.groupBy("customer_id","location").agg(
    count("*").alias("transactions_count"),
    round(avg("amount"),2).alias("avg_amount")
)

In [0]:
user_top_locations.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(user_top_location_table)


In [0]:
%sql
select * from frauddetection.gold.user_locations limit 10;